# 강의 05 · 실습 1 — 관측성 계측 · (1) 강사 시연


## 1. 문제상황

- 놀이공원 구름월드의 안내 워크플로는 손님의 질문을 받아 FAQ에서 관련 항목을 찾고, 찾은 항목을 근거로 모델이 답을 씁니다.
- 워크플로는 이미 동작하고 있지만, 운영 담당자는 화면에 출력된 답 문장 하나만 봅니다.
- 답이 늦게 나오거나 엉뚱하게 나오면, 검색이 잘못 찾은 것인지 모델이 잘못 쓴 것인지 담당자는 구분할 수 없습니다.
- 호출마다 토큰을 얼마나 썼는지, 비용이 얼마인지도 남지 않습니다.


## 2. 문제와 목표

- **문제**: 워크플로 안의 검색 단계와 모델 호출 단계가 밖에서 보이지 않고, 호출마다의 시간·토큰·비용이 어디에도 남지 않습니다.
- **목표**
  - 워크플로 코드를 한 줄도 고치지 않고 계측을 붙여, 질문 하나가 처리될 때마다 LangSmith에 트레이스 하나가 남게 합니다.
    - 트레이스 하나: 부모 런 `faq_agent`(chain) 아래에 검색 tool 런 `faq_search`와 모델 llm 런 `litellm.completion`이 자식으로 달린 것
  - llm 런에는 모델 이름 메타데이터를 붙여 비용 열이 채워지게 합니다.
    - 메타데이터 두 개: `ls_model_name`(gpt-5.6-luna), `ls_provider`(openai)
- **목표 달성 여부의 판정 기준**:
  - 질문 3개를 처리한 뒤 질문마다 부모 런 하나 아래에 검색 tool 런과 모델 llm 런이 자식으로 달렸다는 줄이 화면에 출력되고,
  - LangSmith 화면의 프로젝트 `sesac-lec05-ex01`에 `faq_agent` 런 3개가 각각 자식 런 2개를 달고 남아 있으며,
  - llm 런의 비용 열이 비어 있지 않은 것을 확인합니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec05_ex01_s1_diagram.svg)


## 4. 단계별 요구사항

1. **추적을 켭니다.**
    - 환경변수 `LANGSMITH_TRACING`을 `"true"`로, `LANGSMITH_PROJECT`를 `"sesac-lec05-ex01"`로 둡니다.
    - `.env`에 `LANGSMITH_API_KEY`가 없으면 안내 문장과 함께 실행을 멈춥니다.
2. **모델 호출을 llm 런으로 기록합니다.**
    - `litellm.completion`을 `traceable(run_type="llm", name="litellm.completion", metadata={"ls_model_name": "gpt-5.6-luna", "ls_provider": "openai"})`로 만든 래퍼(wrapper) 함수로 교체합니다.
    - 메타데이터 두 개가 없으면 LangSmith의 비용 열이 빕니다.
3. **검색을 tool 런으로 기록합니다.**
    - `faq_search`를 `traceable(run_type="tool", name="faq_search")`로 만든 래퍼(wrapper) 함수로 교체합니다.
    - 서비스 코드 안의 호출은 이름을 통해 이루어지므로 이름만 교체하면 서비스가 래퍼(wrapper) 함수를 부릅니다.
4. **워크플로를 부모 런으로 승격합니다.**
    - `ask(question)` 함수를 `@traceable(run_type="chain", name="faq_agent")`로 정의하고, 안에서 서비스의 `run_faq_agent`를 부른 뒤 `get_current_run_tree()`로 현재 런의 자식 런 이름과 타입을 화면에 출력하고 답을 돌려줍니다.
    - 출력 형식은 「트리: 부모 런 이름(chain) → 자식 런 이름(tool) · 자식 런 이름(llm)」입니다.
5. **질문 3개를 실행하고 런을 보냅니다.**
    - 환불·인사·야간개장 질문을 차례로 `ask`에 넣어 답 앞부분과 걸린 시간을 출력하고, 마지막에 `Client().flush()`로 남은 런을 서버에 보냅니다.


## 5. 코드 골격 — 관측 계측 4단

이미 도는 워크플로에 계측을 붙이는 순서는 다음 네 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 네 단계와 하나씩 대응합니다. 네 단계 어디에서도 서비스 코드를 고치지 않습니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 환경변수 | 추적을 켜고 프로젝트 이름을 정합니다 | `os.environ["LANGSMITH_TRACING"]`, `os.environ["LANGSMITH_PROJECT"]` | 1 |
| ② llm 런 기록 | 모델 호출 함수를 llm 런으로 감싸 교체합니다 | `traceable(run_type="llm", metadata={...})` | 2 |
| ③ tool 런 기록 | 검색 함수를 tool 런으로 감싸 교체합니다 | `traceable(run_type="tool")` | 3 |
| ④ 부모 런 승격 | 워크플로 전체를 chain 런으로 묶고 실행합니다 | `@traceable(run_type="chain")`, `get_current_run_tree()`, `Client().flush()` | 4, 5 |


## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 키를 읽습니다. 이 실습의 모델 호출은 `litellm.completion`을 직접 씁니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- 이 실습은 OpenAI 키와 LangSmith 키를 함께 씁니다. `.env` 파일에는 다음 두 줄을 넣습니다.
- `litellm.suppress_debug_info = True`는 오류가 났을 때 litellm이 화면에 출력하는 안내 배너를 끕니다. 동작에는 영향이 없습니다.

```
OPENAI_API_KEY=발급받은_키
LANGSMITH_API_KEY=발급받은_키
```


In [1]:
import os
import time
import warnings

from dotenv import load_dotenv, find_dotenv

import litellm
from langsmith import Client, traceable
from langsmith.run_helpers import get_current_run_tree

warnings.filterwarnings("ignore", message="Pydantic serializer warnings")
litellm.suppress_debug_info = True

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

MODEL = "openai/gpt-5.6-luna"
# 같은 OPENAI 키로 호출되는 대체 모델(2026-09-05 확인): openai/gpt-4o-mini · openai/gpt-4.1-mini · openai/gpt-5-mini · openai/gpt-5.4-mini
print("준비를 마쳤습니다. 모델:", MODEL)

준비를 마쳤습니다. 모델: openai/gpt-5.6-luna


계측을 붙일 안내 워크플로입니다. 이미 만들어 둔 서비스이며, 이 실습에서는 고치지 않습니다. `faq_search`가 질문과 겹치는 낱말이 많은 FAQ 항목을 찾고, `run_faq_agent`가 찾은 항목을 근거로 모델을 한 번 불러 답을 돌려줍니다. 워크플로가 도는 것을 먼저 확인합니다.


In [2]:
FAQ_ITEMS = [
    {"category": "환불", "question": "자유이용권 환불 규정 알려 주세요",
     "answer": "이용일 전날까지 취소하면 전액 환불됩니다. 이용일 당일 취소는 50%만 환불됩니다. 입장 뒤에는 환불되지 않습니다."},
    {"category": "운영", "question": "운영 시간이 어떻게 되나요?",
     "answer": "평일은 10시부터 19시까지, 주말은 10시부터 21시까지 운영합니다."},
    {"category": "야간", "question": "야간개장은 언제 하나요?",
     "answer": "금요일과 토요일에는 22시까지 야간개장을 합니다. 야간개장 날에는 20시 30분에 야간 퍼레이드가 있습니다."},
    {"category": "주차", "question": "주차 요금은 얼마인가요?",
     "answer": "자유이용권 소지자는 4시간까지 무료이고, 그 뒤로는 시간당 2,000원입니다."},
]


def faq_search(question: str) -> list:
    """질문과 겹치는 낱말이 많은 FAQ 항목을 최대 2개 돌려준다 (검색 도구)."""
    words = set(question.replace("?", " ").split())
    scored = []
    for item in FAQ_ITEMS:
        text = item["category"] + " " + item["question"] + " " + item["answer"]
        overlap = sum(1 for w in words if w[:2] in text)
        scored.append((overlap, item))
    scored.sort(key=lambda pair: pair[0], reverse=True)
    return [f"[{it['category']}] Q: {it['question']} A: {it['answer']}" for score, it in scored[:2] if score > 0]


def run_faq_agent(question: str) -> str:
    """검색 도구로 근거를 찾고, 근거를 시스템 메시지에 넣어 모델을 한 번 부른다."""
    hits = faq_search(question)
    system = ("너는 놀이공원 구름월드의 안내 담당자다. 아래 근거에 적힌 내용만으로 두 문장 안에서 답한다. "
              "인사말에는 짧은 인사로 답한다. 근거에 없는 내용은 '해당 내용은 확인할 수 없습니다.'라고만 답한다.\n"
              "=== 근거 ===\n" + "\n".join(hits))
    res = litellm.completion(model=MODEL, messages=[{"role": "system", "content": system},
                                                    {"role": "user", "content": question}])
    return res.choices[0].message.content.strip()


print(run_faq_agent("주차 요금은 얼마인가요?"))

자유이용권 소지자는 4시간까지 무료이며, 이후에는 시간당 2,000원입니다.


### 단계 ① — 환경변수 (요구사항 1)

추적은 환경변수 두 개로 켭니다. `LANGSMITH_TRACING`이 `"true"`이면 `traceable`로 만든 래퍼(wrapper) 함수의 실행이 런으로 기록되고, `LANGSMITH_PROJECT`가 런이 쌓일 프로젝트 이름입니다. LangSmith 키가 없으면 런을 보낼 곳이 없으므로 여기서 멈춥니다.


In [3]:
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "sesac-lec05-ex01"
if not os.environ.get("LANGSMITH_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 LANGSMITH_API_KEY 한 줄을 넣습니다.")

print("추적:", os.environ["LANGSMITH_TRACING"], "· 프로젝트:", os.environ["LANGSMITH_PROJECT"])

추적: true · 프로젝트: sesac-lec05-ex01


### 단계 ② — llm 런 기록 (요구사항 2)

`traceable`은 함수를 받아 「실행을 런으로 기록하는 함수」를 돌려줍니다. `run_type="llm"`으로 감싸면 모델 호출로 기록됩니다. 서비스는 `litellm.completion`이라는 이름으로 모델을 부르므로, 그 이름에 래퍼(wrapper) 함수를 다시 묶어 두면 서비스 코드를 고치지 않고도 모든 모델 호출이 llm 런이 됩니다. `ls_model_name`·`ls_provider` 메타데이터는 LangSmith가 비용을 계산할 때 쓰는 이름표입니다.


In [4]:
litellm.completion = traceable(
    run_type="llm",
    name="litellm.completion",
    metadata={"ls_model_name": "gpt-5.6-luna", "ls_provider": "openai"},
)(litellm.completion)

print("llm 런 기록:", litellm.completion.__name__)

llm 런 기록: completion


### 단계 ③ — tool 런 기록 (요구사항 3)

같은 방법으로 검색 함수를 `run_type="tool"`로 감싸 이름에 다시 묶습니다. `run_faq_agent`는 `faq_search`라는 이름을 실행 시점에 찾으므로, 이름이 래퍼(wrapper) 함수를 가리키면 서비스가 래퍼(wrapper) 함수를 부릅니다.


In [ ]:

faq_search = traceable(run_type="tool", name="faq_search")(faq_search)

print("tool 런 기록:", faq_search.__name__)

tool 런 기록: faq_search


### 단계 ④ — 부모 런 승격 (요구사항 4, 5)

워크플로 전체를 `run_type="chain"`으로 만든 래퍼(wrapper) 함수 안에서 부르면, 그 안에서 실행된 tool 런과 llm 런이 자식으로 달립니다. `get_current_run_tree()`는 지금 실행 중인 런을 돌려주므로, 자식 런의 이름과 타입을 화면에 출력해 트리가 실제로 만들어졌는지 확인합니다. 런은 뒤에서 모아 보내므로 마지막에 `Client().flush()`로 남은 런을 서버에 보냅니다.


In [ ]:
@traceable(run_type="chain", name="faq_agent")
def ask(question: str) -> str:
    """워크플로를 부모 런 안에서 부르고, 자식 런의 이름과 타입을 출력한다."""
    answer = run_faq_agent(question)
    run = get_current_run_tree()
    children = " · ".join(f"{c.name}({c.run_type})" for c in run.child_runs)
    print(f"  트리: {run.name}({run.run_type}) → {children}")
    return answer


QUESTIONS = ["자유이용권 환불이 되나요?", "안녕하세요!", "야간개장은 몇 시까지인가요?"]

for question in QUESTIONS:
    started = time.perf_counter()
    answer = ask(question)
    print(f"[{question}] {answer[:50]}  ({time.perf_counter() - started:.2f}초)")
    print()

Client().flush()
print("런 전송을 마쳤습니다. LangSmith 프로젝트 sesac-lec05-ex01에서 확인합니다.")

  트리: faq_agent(chain) → faq_search(tool) · litellm.completion(llm)
[자유이용권 환불이 되나요?] 이용일 전날까지 취소하면 전액 환불됩니다. 이용일 당일 취소는 50%만 환불되며, 입장 뒤  (1.29초)

  트리: faq_agent(chain) → faq_search(tool) · litellm.completion(llm)
[안녕하세요!] 안녕하세요! 天天好  (1.42초)

  트리: faq_agent(chain) → faq_search(tool) · litellm.completion(llm)
[야간개장은 몇 시까지인가요?] 금요일과 토요일에는 22시까지 야간개장을 합니다.  (1.21초)

런 전송을 마쳤습니다. LangSmith 프로젝트 sesac-lec05-ex01에서 확인합니다.


## 7. 실행 결과 확인

위 실행 결과에서 다음 세 가지를 확인합니다.

1. 단계 ④ 출력에 질문마다 「트리: faq_agent(chain) → faq_search(tool) · litellm.completion(llm)」이 출력됩니다. 부모 런 하나에 자식 런 두 개가 달렸다는 뜻입니다.
2. 질문마다 답 앞부분과 걸린 시간이 출력됩니다. 시간의 대부분은 llm 런이 차지합니다.
3. LangSmith 화면의 프로젝트 `sesac-lec05-ex01`에 `faq_agent` 런 3개가 남고, 펼치면 `faq_search`와 `litellm.completion`이 자식으로 보입니다. `litellm.completion` 런의 토큰과 비용 열이 비어 있지 않습니다.

서비스 코드는 한 줄도 고치지 않았습니다. 이름을 다시 묶는 것만으로 트레이스가 생긴 것이 관측 계측의 핵심입니다.
